# Adaboost

AdaBoost or Adaptive Boosting is an ensemble boosting classifier and it was proposed 
by Yoav Freund and Robert Schapire in 1996.

This algorithm combines multiple weak classifiers (mostly decision trees) to increase the accuracy of the individual classifiers. The basic concept behind Adaboost is to construct a sequence of weighted classifiers on the training set.

### Advantages: 

- They are very strong classifiers with error rate close to zero.

- They allow you to identify the model which failed the accurate prediction.

- They are less affected by the overfitting problem.



### Adaboost algorithm steps

With:

- Samples $x_1 \dots x_n$ each in $\mathbb{R}^d$
- Desired outputs $ y_1 \dots y_n, y \in \{-1, 1\}$
- Initial weights $w_{{1,1}}\dots w_{{n,1}} \;\text{set to}\; {\frac {1}{n}}$. Here $w_{1,t}$ where $t$ is the iteration index; $t=1$ is the first or initial iteration.
- Error function $ E(f(x), y, i) = e^{-y_i f(x_i)}$
- Weak learners ${\displaystyle h\colon x\rightarrow \{-1,1\}}$
    
For $t \;\text{in}\; 1 \dots T:$
  - Choose $h_t(x)$:
      - Find weak learner $h_t(x)$ that minimizes $\epsilon_{t}$, the weighted sum error for misclassified points
      ${\displaystyle \epsilon _{t}=\sum _{\stackrel {i=1}{h_{t}(x_{i})\neq y_{i}}}^{n}w_{i,t}}$
      - Choose $\alpha_t = \frac{1}{2} \ln \left(\frac{1-\epsilon_t}{\epsilon_t}\right)$ 
   - Add to ensemble:
        - $ F_t(x) = F_{t-1}(x) + \alpha_t h_t(x)$
   - Update weights:
        - $w_{i,t+1} = w_{i,t} e^{-y_i \alpha_t h_t(x_i)} \text{for $i$ in }{\displaystyle 1\dots n}$
        
        - Renormalize $w_{i,t+1} \;\text{such that}\sum_i w_{i,t+1} = 1$


### Adaboost implementation in Python

In [ ]:
import warnings
warnings.filterwarnings(action='ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

In [ ]:
# ------------------------------------------------------------------
# Building blocks of (discrete) AdaBoost, with y in {-1, +1}
# ------------------------------------------------------------------

def compute_error(y, y_pred, w_i):
    '''
    Weighted error rate of the current weak learner:

        epsilon_t = sum_{i : h_t(x_i) != y_i} w_i      (weights sum to 1)

    y      : true labels, shape (n,), values in {-1, +1}
    y_pred : predictions of the current weak learner, shape (n,)
    w_i    : current sample weights, shape (n,)
    '''
    # TODO: weighted fraction of misclassified points
    error = # complete
    return error


def compute_alpha(error):
    '''
    Voting weight (confidence) given to the current weak learner:

        alpha_t = 1/2 * ln( (1 - epsilon_t) / epsilon_t )

    error : weighted error rate of the weak learner (compute_error's output)
    '''
    # TODO: alpha_t as a function of the weighted error
    alpha = # complete
    return alpha


def update_weights(w_i, alpha, y, y_pred):
    '''
    Reweight the training points after one boosting round and renormalize
    so the weights sum back to 1:

        w_i <- w_i * exp(-alpha * y_i * h_t(x_i))
        w_i <- w_i / sum(w_i)

    w_i    : current sample weights, shape (n,)
    alpha  : voting weight of the weak learner just trained
    y      : true labels, values in {-1, +1}
    y_pred : predictions of that weak learner
    '''
    # TODO: exponential reweighting + renormalization
    weights = # complete
    return weights

In [ ]:
def fit(X, y, T=10):
    '''
    Train an AdaBoost ensemble of decision stumps.

    X : independent variables, shape (n_samples, n_features)
    y : target variable, values in {-1, +1}, shape (n_samples,)
    T : number of boosting rounds (= number of weak classifiers)

    Returns
    -------
    DTC_list        : list of T fitted stumps (DecisionTreeClassifier, max_depth=1)
    alphas          : list of T voting weights alpha_t
    training_errors : list of T weighted training errors epsilon_t
    '''
    alphas = []
    training_errors = []
    DTC_list = []  # the T weak classifiers (stumps)

    for t in range(T):
        # --- 1) sample weights for this round ------------------------
        if t == 0:
            # all points start with equal weight 1/n
            w_i = # complete
        else:
            # reweight using the previous round's learner and alpha
            w_i = update_weights(w_i, alpha_m, y, y_pred)

        # --- 2) train a weak learner (stump) with those weights -------
        DTC = DecisionTreeClassifier(max_depth=1)  # stump: a single split
        DTC.fit(X, y, sample_weight=w_i)
        y_pred = DTC.predict(X)
        DTC_list.append(DTC)

        # --- 3) weighted error and voting weight of this stump --------
        error_m = compute_error(y, y_pred, w_i)
        training_errors.append(error_m)

        alpha_m = compute_alpha(error_m)
        alphas.append(alpha_m)

    return DTC_list, alphas, training_errors

In [ ]:
def predict(X, DTC_list, alphas):
    '''
    Combine the weak classifiers into the final AdaBoost prediction:

        F(x) = sign( sum_t alpha_t * h_t(x) )

    X        : independent variables, shape (n_samples, n_features)
    DTC_list : list of fitted weak classifiers (fit's output)
    alphas   : list of their voting weights (fit's output)
    '''
    T = len(DTC_list)  # number of weak classifiers actually trained

    # weak_preds[:, t] = alpha_t * h_t(x) for every point
    weak_preds = pd.DataFrame(index=range(len(X)), columns=range(T))
    for t in range(T):
        weak_preds.iloc[:, t] = DTC_list[t].predict(X) * alphas[t]

    # final vote: sign of the weighted sum of all weak predictions
    y_pred = np.sign(weak_preds.sum(axis=1)).astype(int)
    return y_pred

### Classification example

In [ ]:
# ------------------------------------------------------------------
# Synthetic binary classification dataset
# ------------------------------------------------------------------
X, y = make_classification(
    n_samples=500,      # 500 observations
    n_features=5,       # 5 total features
    n_informative=2,    # 2 of them actually useful
    n_classes=2,        # binary target
    random_state=999,   # for reproducibility
)
y[y == 0] = -1  # AdaBoost expects labels in {-1, +1}, not {0, 1}

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)

In [ ]:
# ------------------------------------------------------------------
# Train the AdaBoost ensemble and predict on the held-out test set
# ------------------------------------------------------------------
T = 10  # number of weak classifiers (boosting rounds)
DTC_list, alphas, training_errors = fit(X_train, y_train, T)
y_pred = predict(X_test, DTC_list, alphas)

In [ ]:
# ------------------------------------------------------------------
# Evaluate on the test set
# ------------------------------------------------------------------
# Note: MSE on {-1, +1} labels isn't a typical classification metric
# (it just rescales the misclassification count by 4); kept here only
# as a quick sanity check alongside accuracy.
mse = np.sum((y_test - y_pred) ** 2) / len(y_test)
print('MSE is : %2.2f' % mse)

acc = np.sum(np.equal(y_test, y_pred).astype(float)) / len(y_test)
print('Accuracy is : %2.2f' % acc)

In [ ]:
# ------------------------------------------------------------------
# Training error across boosting rounds
# ------------------------------------------------------------------
plt.plot(training_errors, '-r')
plt.title('Error history')
plt.xlabel('iterations')
plt.ylabel('error')
plt.grid()

In [ ]:
# ------------------------------------------------------------------
# Baseline: a single, unrestricted decision tree (no boosting)
# ------------------------------------------------------------------
DTCNB = DecisionTreeClassifier(max_depth=None)  # full tree, not a stump
DTCNB.fit(X_train, y_train)
y_predNB = DTCNB.predict(X_test)

accNB = np.sum(np.equal(y_test, y_predNB).astype(float)) / len(y_test)
print('Accuracy is : %2.2f' % accNB)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
plot_tree(
    decision_tree=DTCNB,
    filled=True,
    impurity=True,
    ax=ax,
)

# Analysis of a simple example

In [ ]:
# Five simple 1-D training points, used below to see how sample_weight
# changes the split and the Gini impurity of a single stump.
X = [[0], [1], [2], [3], [4]]  # 5 training examples
Y = [1, 2, 1, 1, 2]            # class labels

## Without weighting

In [ ]:
dtc = DTC(max_depth=1)
dtc.fit(X, Y)

gini = $\sum_{clases} 1 -p(x)^2$

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
plot_tree(
    label='all',
    decision_tree=dtc,
    filled=True,
    impurity=True,
    ax=ax,
)

In [ ]:
print('thresholds = %f, %f, %f' % (
    dtc.tree_.threshold[0], dtc.tree_.threshold[1], dtc.tree_.threshold[2]))
print('impurities = %f, %f, %f' % (
    dtc.tree_.impurity[0], dtc.tree_.impurity[1], dtc.tree_.impurity[2]))

### Some manual calculations

In [ ]:
# Manual Gini-impurity check (unweighted), matching the tree above:
#   parent node : 3 of class 1, 2 of class 2  -> p = 3/5, 2/5
#   left leaf   : 3 of class 1, 1 of class 2  -> p = 3/4, 1/4
#   right leaf  : all class 2                 -> pure, gini = 0
parent_gini = 1 - (3.0 / 5.0) ** 2 - (2.0 / 5.0) ** 2
print('gini value of parent node = %f' % parent_gini)

leaf1_gini = 1 - (3.0 / 4.0) ** 2 - (1.0 / 4.0) ** 2
print('gini value of leaf1 node = %f' % leaf1_gini)

leaf2_gini = 1 - (0.0 / 1.0) ** 2 - (1.0 / 1.0) ** 2
print('gini value of leaf2 node = %f' % leaf2_gini)

## With weighting

In [ ]:
# Re-fit the same stump, now giving some points more weight than others
# (point weights = [1, 3, 1, 2, 1]) -- compare the resulting threshold
# and impurities with the unweighted case above.
dtc.fit(X, Y, sample_weight=[1, 3, 1, 2, 1])
print(dtc.tree_.threshold)
print(dtc.tree_.impurity)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
plot_tree(
    decision_tree=dtc,
    filled=True,
    impurity=True,
    ax=ax,
)

In [ ]:
# Manual Gini-impurity check (weighted), matching the tree above.
# Total weight = 1+3+1+2+1 = 8; the split now separates the points by
# weight, not by raw count.
parent_gini = 1 - (4.0 / 8.0) ** 2 - (4.0 / 8.0) ** 2
print('gini value of parent node = %f' % parent_gini)

leaf1_gini = 1 - (1.0 / 4.0) ** 2 - (3.0 / 4.0) ** 2
print('gini value of leaf1 node = %f' % leaf1_gini)

leaf2_gini = 1 - (3.0 / 4.0) ** 2 - (1.0 / 4.0) ** 2
print('gini value of leaf2 node = %f' % leaf2_gini)